Cài đặt

In [116]:
!pip install folium


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


Code hiện bản đồ điểm đạc gữa mặc định là tp.HCM

In [117]:
import folium

def show_map():
    m = folium.Map(
        location=[10.8231, 106.6297],
        zoom_start=13
    )
    return m

In [118]:
show_map()

In [119]:
def show_rooms_ipynb():
    m = folium.Map(location=[10.82, 106.63], zoom_start=13)

   

    for name, lat, lng in rooms:
        folium.Marker(
            [lat, lng],
            popup=name
        ).add_to(m)

    return m

In [120]:
rooms = [
        ("Phòng A", 10.823, 106.63),
        ("Phòng B", 10.819, 106.627)
    ]

In [121]:
show_rooms_ipynb()

Tính khoảng cách giữa 2 điểm GPS theo công thức Haversine
Đơn vị trả về: km

In [122]:
import folium
import math

def haversine_km(lat1, lon1, lat2, lon2):
    
    R = 6371  

    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))

    return R * c

1 điểm người dùng chọn

In [123]:
center = {
    "name": "Điểm chọn test",
    "lat": 10.7540,   # Quận 5
    "lng": 106.6636
}

danh sách mẫu phòng trọ chọn


In [124]:
center = {
    "name": "Điểm chọn test",
    "lat": 10.7540,   # Quận 5
    "lng": 106.6636
}
rooms = [
    ("Phòng Quận 5", 10.7548, 106.6630),
    ("Phòng Quận 10", 10.7769, 106.6671),
    ("Phòng 10", 10.7769, 106.7009),
   # ("Phòng 0", 10.7540, 106.6636),
    ("Phòng Thủ Đức", 10.8506, 106.7719)
]

lấy điểm center và tìm trọ gần 5km trong danh sách


In [ ]:
def test_1_point_find_rooms():
    # Tạo bản đồ
    m = folium.Map(
        location=[center["lat"], center["lng"]],
        zoom_start=13,
        #tiles="OpenStreetMap",
        #width="100%",
        #height="600px"
    )

    # 📍 Marker điểm chọn
    folium.Marker(
        [center["lat"], center["lng"]],
        popup="📍 Điểm người dùng chọn",
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(m)

    # 🔵 Vòng bán kính 5km
    folium.Circle(
        location=[center["lat"], center["lng"]],
        radius=5000,   # mét
        color="blue",
        fill=True,
        fill_opacity=0.2,
        popup="Bán kính 5km"
    ).add_to(m)

    # 🏠 Các phòng trọ trong 5km
    for ten, lat, lng in rooms:
        d = haversine_km(center["lat"], center["lng"], lat, lng)

        if d <= 5:
            folium.Marker(
                [lat, lng],
                popup=f"🏠 {ten}<br>📏 {d:.2f} km",
                icon=folium.Icon(color="green", icon="home")
            ).add_to(m)

    # ⭐ TỰ ĐỘNG PHÓNG TO / CĂN KHUNG
    m.fit_bounds(m.get_bounds())

    return m

In [126]:
test_1_point_find_rooms()

Tool

In [127]:
center = {"lat": 10.7540, "lng": 106.6636}

rooms = [
    ("Phòng Quận 5", 10.7548, 106.6630),
    ("Phòng Quận 10", 10.7769, 106.6671),
    ("Phòng 10", 10.7769, 106.7009),
    ("Phòng 0", 10.7540, 106.6636),
    ("Phòng Thủ Đức", 10.8506, 106.7719)
]

## VẼ TUYẾN ĐƯỜNG ORS

Tool

In [3]:
import requests

ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImYwNDc5MjUxOTFmZjQ0ZWViN2I3YzBhNmZmOWViN2YwIiwiaCI6Im11cm11cjY0In0="

def draw_route(start, end, vehicle="car"):
    """
    start, end: (lat, lng)
    return: feature GeoJSON hoặc None
    """

    # 1️⃣ Chuẩn hóa profile
    profile = "foot-walking" if vehicle == "walk" else "driving-car"

    # 2️⃣ Tạo URL
    url = f"https://api.openrouteservice.org/v2/directions/{profile}"

    params = {
        "start": f"{start[1]},{start[0]}",  # lng,lat
        "end": f"{end[1]},{end[0]}"
    }

    headers = {
        "Authorization": f"Bearer {ORS_API_KEY}"
    }

    # 3️⃣ Gửi request
    resp = requests.get(url, headers=headers, params=params)

    if resp.status_code != 200:
        print("❌ ORS ERROR:", resp.text)
        return None

    data = resp.json()

    if "features" not in data:
        print("❌ Không có tuyến đường")
        return None

    return data["features"][0]

Hiện Bản đồ

In [5]:
import folium

def show_route_map(start, end):
    route = draw_route(start, end)

    if route is None:
        print("❌ Không vẽ được tuyến – kiểm tra API")
        return

    # Lấy geometry
    coords = route["geometry"]["coordinates"]

    # Lấy khoảng cách
    distance_km = route["properties"]["segments"][0]["distance"] / 1000

    # Đảo [lng,lat] → [lat,lng]
    route_latlng = [(lat, lng) for lng, lat in coords]

    # Tạo bản đồ
    m = folium.Map(location=start, zoom_start=13)

    # Vẽ tuyến
    folium.PolyLine(
        route_latlng,
        color="blue",
        weight=5,
        tooltip=f"{distance_km:.2f} km"
    ).add_to(m)

    # Marker
    folium.Marker(start, tooltip="Start").add_to(m)
    folium.Marker(end, tooltip="End").add_to(m)

    return m

In [ ]:
start = (10.7540, 106.6636)  # Quận 5
end   = (10.7769, 106.7009)  # Quận 1

show_route_map(start, end)

Tính thời gian

In [14]:
import requests

ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImYwNDc5MjUxOTFmZjQ0ZWViN2I3YzBhNmZmOWViN2YwIiwiaCI6Im11cm11cjY0In0="

def get_route_info(start, end, vehicle="car"):
    """
    Trả về:
    - distance_km
    - duration_min
    - geometry (để vẽ tuyến)
    """

    profile_map = {
        "car": "driving-car",
        "moto": "driving-car",
        "walk": "foot-walking",
        "bike": "cycling-regular"
    }

    profile = profile_map.get(vehicle, "driving-car")

    url = f"https://api.openrouteservice.org/v2/directions/{profile}"

    params = {
        "start": f"{start[1]},{start[0]}",  # lng, lat
        "end": f"{end[1]},{end[0]}"
    }

    headers = {
        "Authorization": ORS_API_KEY
    }

    resp = requests.get(url, headers=headers, params=params).json()

    if "features" not in resp:
        return None

    feature = resp["features"][0]
    summary = feature["properties"]["segments"][0]

    return {
        "distance_km": summary["distance"] / 1000,
        "duration_min": summary["duration"] / 60,
        "geometry": feature["geometry"]["coordinates"]
    }

In [15]:
import folium

def show_route_map(start, end, vehicle="car"):
    route = get_route_info(start, end, vehicle)

    if route is None:
        print("❌ Không tìm được tuyến đường")
        return

    m = folium.Map(location=start, zoom_start=13)

    # Marker điểm đầu
    folium.Marker(
        start,
        popup="📍 Điểm bắt đầu",
        icon=folium.Icon(color="green")
    ).add_to(m)

    # Marker điểm cuối
    folium.Marker(
        end,
        popup="🏁 Điểm đến",
        icon=folium.Icon(color="red")
    ).add_to(m)

    # ORS trả về [lng, lat] → đảo lại
    route_latlng = [(lat, lng) for lng, lat in route["geometry"]]

    folium.PolyLine(
        route_latlng,
        color="blue",
        weight=5,
        popup=(
            f"📏 {route['distance_km']:.2f} km<br>"
            f"⏱️ {route['duration_min']:.1f} phút"
        )
    ).add_to(m)

    m.fit_bounds(m.get_bounds())

    return m

In [16]:
start = (10.7540, 106.6636)  # Quận 5
end   = (10.7769, 106.7009)  # Quận 1

show_route_map(start, end, vehicle="moto")